In [75]:
import pandas as pd

Đọc 3 bảng dữ liệu chính cần thiết cho việc phân tích doanh thu

In [76]:
sales = pd.read_csv('sales.csv')
product = pd.read_csv('products.csv')
stores = pd.read_csv('stores.csv')

Số dòng gốc 

In [77]:
so_dong_goc = len(sales)
print(f"Tổng số giao dịch ban đầu: {so_dong_goc:,} dòng")

Tổng số giao dịch ban đầu: 829,262 dòng


Gộp bảng 

In [78]:
df_merged = pd.merge(sales, product, on='Product_ID', how='left')
df_merged = pd.merge(df_merged, stores, on='Store_ID', how='left')



Sanity Check

In [79]:
so_dong_sau_gop = len(df_merged)
print(f"Số dòng sau khi gộp: {so_dong_sau_gop:,} dòng")

if so_dong_goc == so_dong_sau_gop:
    print("✅ AN TOÀN: Dữ liệu khớp hoàn toàn, bảo toàn 100% hóa đơn!")
else:
    print("❌ CẢNH BÁO: Số dòng bị lệch, dữ liệu đang bị phình to hoặc rớt mất.")

Số dòng sau khi gộp: 829,262 dòng
✅ AN TOÀN: Dữ liệu khớp hoàn toàn, bảo toàn 100% hóa đơn!


Kiểm tra dữ liệu bị thiếu trước khi đưa vào tín toán

In [80]:
print("Kiểm tra dữ liệu bị thiếu (Missing values):")
print(df_merged.isnull().sum())
print("-" * 40)

Kiểm tra dữ liệu bị thiếu (Missing values):
Sale_ID             0
Date                0
Store_ID            0
Product_ID          0
Units               0
Product_Name        0
Product_Category    0
Product_Cost        0
Product_Price       0
Store_Name          0
Store_City          0
Store_Location      0
Store_Open_Date     0
dtype: int64
----------------------------------------


Tính toán và thu lợi nhuận

In [81]:
# Chuyển đổi định dạng giá (bỏ dấu $) sang số float
df_merged['Product_Price'] = df_merged['Product_Price'].str.replace('$', '', regex=False).str.strip().astype(float)
df_merged['Product_Cost'] = df_merged['Product_Cost'].str.replace('$', '', regex=False).str.strip().astype(float)

# Tạo cột tính toán mới: Doanh thu (Revenue) và Lợi nhuận (Profit)
df_merged['Revenue'] = df_merged['Units'] * df_merged['Product_Price']
df_merged['Profit'] = df_merged['Units'] * (df_merged['Product_Price'] - df_merged['Product_Cost'])

df_merged.head()


,Sale_ID,Date,Store_ID,Product_ID,Units,Product_Name,Product_Category,Product_Cost,Product_Price,Store_Name,Store_City,Store_Location,Store_Open_Date,Revenue,Profit
0,1,2022-01-01,24,4,1,Chutes & Ladders,Games,9.99,12.99,Maven Toys Aguascalientes 1,Aguascalientes,Downtown,2010-07-31,12.99,3.0
1,2,2022-01-01,28,1,1,Action Figure,Toys,9.99,15.99,Maven Toys Puebla 2,Puebla,Downtown,2011-04-01,15.99,6.0
2,3,2022-01-01,6,8,1,Deck Of Cards,Games,3.99,6.99,Maven Toys Mexicali 1,Mexicali,Commercial,2003-12-13,6.99,3.0
3,4,2022-01-01,48,7,1,Dart Gun,Sports & Outdoors,11.99,15.99,Maven Toys Saltillo 2,Saltillo,Commercial,2016-03-23,15.99,4.0
4,5,2022-01-01,44,18,1,Lego Bricks,Toys,34.99,39.99,Maven Toys Puebla 3,Puebla,Residential,2014-12-27,39.99,5.0


Xuất dữ liệu đã làm sạch

In [82]:
df_merged.to_csv('cleaned_maven_toys_sales.csv', index=False)
print("🚀 Đã xuất file thành công!")

🚀 Đã xuất file thành công!


In [83]:
import sqlite3
import pandas as pd

# Khởi tạo môi trường cơ sở dữ liệu nội bộ (In-memory Database)
conn = sqlite3.connect(':memory:')

# Tải dữ liệu đã làm sạch vào hệ thống SQL
df_clean = pd.read_csv('cleaned_maven_toys_sales.csv')
df_clean.to_sql('sales_data', conn, index=False, if_exists='replace')

print("✅ Triển khai Database thành công! Đang truy vấn báo cáo...\n")

#Khai báo truy vấn nghiệp vụ (SQL Query)
query = """
SELECT 
    Product_Category,
    SUM(Revenue) AS Total_Revenue,
    SUM(Profit) AS Total_Profit
FROM sales_data
GROUP BY Product_Category
ORDER BY Total_Profit DESC;
"""

# Thực thi truy vấn và xuất báo cáo
report_df = pd.read_sql_query(query, conn)
display(report_df)

✅ Triển khai Database thành công! Đang truy vấn báo cáo...



,Product_Category,Total_Revenue,Total_Profit
0,Toys,5093241.00,1079527.0
1,Electronics,2246771.25,1001437.0
2,Art & Crafts,2705364.26,753354.0
3,Games,2226836.27,673993.0
4,Sports & Outdoors,2172359.57,505718.0
